# 节点 03：反向传播——20 行代码解决 XOR

节点 02 里，感知机在 XOR 面前永远失败。
这个 notebook 要展示：**加一个隐藏层 + 反向传播，4 个 XOR 样本全部学会。**

所有代码从零实现，不使用任何机器学习库。

## Cell 1：准备工具

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 固定随机种子，保证每次结果一样
np.random.seed(0)

print('工具准备好了。')
print('NumPy version:', np.__version__)


工具准备好了。
NumPy version: 1.26.4


## Cell 2：XOR 数据——还是那 4 行

这是节点 02 的老面孔。感知机学不会，我们要用新方法解决它。

In [2]:
# XOR 的全部 4 个样本
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)

y = np.array([[0],
              [1],
              [1],
              [0]], dtype=float)  # 列向量，方便矩阵运算

print('输入 X:')
print(X)
print('\n标签 y:')
print(y.T)  # 转置打印更清楚

输入 X:
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]

标签 y:
[[0. 1. 1. 0.]]


## Cell 3：Sigmoid——「软版开关」

感知机用阶跃函数（结果只有 0 或 1，跳变很突然）。
反向传播需要「光滑」的激活函数，因为要求导数。

Sigmoid 函数把任何数压到 (0, 1) 之间，过渡平滑：

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

它的导数有一个神奇的简化：

$$\sigma'(z) = \sigma(z) \times (1 - \sigma(z))$$

已经知道 $\sigma(z)$ 了，导数几乎不需要额外计算。

In [3]:
def sigmoid(z):
    """Sigmoid 激活函数。将任意实数压到 (0, 1)，使用 clip 防止数值溢出。"""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(a):
    """Sigmoid 的导数。输入 a = sigmoid(z)，不是 z 本身。"""
    return a * (1.0 - a)

# 验证几个值
test_z = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])
test_a = sigmoid(test_z)
print('z 值        :', test_z)
print('sigmoid(z)  :', np.round(test_a, 4))
print('导数 σ(1-σ)  :', np.round(sigmoid_deriv(test_a), 4))
print()
print('验证：σ(0) 应该精确等于 0.5:', sigmoid(0.0))


z 值        : [-3. -1.  0.  1.  3.]
sigmoid(z)  : [0.0474 0.2689 0.5    0.7311 0.9526]
导数 σ(1-σ)  : [0.0452 0.1966 0.25   0.1966 0.0452]

验证：σ(0) 应该精确等于 0.5: 0.5


## Cell 4：网络结构——搭积木

我们要搭一个最简单的两层网络：

```
输入层 (2个节点)  →  隐藏层 (2个节点)  →  输出层 (1个节点)
     A, B              h1, h2                  ŷ
```

参数总数：
- W1：输入层→隐藏层的权重矩阵，形状 (2×2)
- b1：隐藏层偏置，形状 (2,)
- W2：隐藏层→输出层的权重矩阵，形状 (2×1)
- b2：输出层偏置，形状 (1,)

In [4]:
# 随机初始化权重（小随机值，不能全为零）
# 全零初始化会让所有隐藏节点学到完全一样的东西
np.random.seed(0)  # 固定种子，保证可重复
W1 = np.random.randn(2, 2) * 0.5   # (输入维度 × 隐藏节点数)
b1 = np.zeros((1, 2))               # 偏置初始化为 0
W2 = np.random.randn(2, 1) * 0.5   # (隐藏节点数 × 输出维度)
b2 = np.zeros((1, 1))

print('W1 形状:', W1.shape, '  (每个输入节点到每个隐藏节点各一条边)')
print('b1 形状:', b1.shape)
print('W2 形状:', W2.shape, '  (每个隐藏节点到输出节点各一条边)')
print('b2 形状:', b2.shape)
print()
print('总参数数量:', W1.size + b1.size + W2.size + b2.size)


W1 形状: (2, 2)   (每个输入节点到每个隐藏节点各一条边)
b1 形状: (1, 2)
W2 形状: (2, 1)   (每个隐藏节点到输出节点各一条边)
b2 形状: (1, 1)

总参数数量: 9


## Cell 5：前向传播——从输入到输出

前向传播就是「把数据从左往右算一遍」：

1. 计算隐藏层的加权输入：$z_1 = X \cdot W_1 + b_1$
2. 激活：$a_1 = \sigma(z_1)$
3. 计算输出层的加权输入：$z_2 = a_1 \cdot W_2 + b_2$
4. 激活得到最终预测：$\hat{y} = \sigma(z_2)$

In [5]:
def forward(X, W1, b1, W2, b2):
    """前向传播：从输入计算预测值，同时保存中间结果（反向传播需要）。"""
    z1 = X @ W1 + b1          # 隐藏层加权求和
    a1 = sigmoid(z1)           # 隐藏层激活值
    z2 = a1 @ W2 + b2          # 输出层加权求和
    a2 = sigmoid(z2)           # 最终预测值 ŷ
    # 返回中间值，反向传播时需要
    return a1, a2

# 测试前向传播
a1, a2 = forward(X, W1, b1, W2, b2)
print('初始化后的预测（还没训练，接近随机）：')
print('输入      预测ŷ    真实y')
for i in range(len(X)):
    print(f'  {X[i]}  →  {a2[i][0]:.4f}   {int(y[i][0])}')

初始化后的预测（还没训练，接近随机）：
输入      预测ŷ    真实y
  [0. 0.]  →  0.5554   0
  [0. 1.]  →  0.5524   1
  [1. 0.]  →  0.5967   1
  [1. 1.]  →  0.5888   0


## Cell 6：损失函数——「错了多少」

我们用均方误差（Mean Squared Error, MSE）衡量预测有多差：

$$L = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

读法：把每个样本的「预测值 - 真实值」平方后取平均。

- 全对时：损失 = 0
- 越错：损失越大

In [6]:
def mse_loss(y_pred, y_true):
    """均方误差损失。"""
    return np.mean((y_pred - y_true) ** 2)

initial_loss = mse_loss(a2, y)
print(f'初始损失（训练前）: {initial_loss:.4f}')
print(f'（随机初始化时，损失大约在 0.2~0.3 之间）')

初始损失（训练前）: 0.2545
（随机初始化时，损失大约在 0.2~0.3 之间）


## Cell 7：反向传播——把「错误」从后往前传

这是核心。用链式法则，从输出层的误差出发，一步步算每个权重该怎么改。

**符号约定**：
- $\delta_2$：输出层的「责任信号」
- $\delta_1$：隐藏层的「责任信号」（由 $\delta_2$ 反传回来）

**推导（每步一句话）**：

1. 输出层误差：$\delta_2 = (\hat{y} - y) \times \sigma'(a_2)$
   — 预测错了多少 × 激活函数的灵敏度

2. 传回隐藏层：$\delta_1 = (\delta_2 \cdot W_2^T) \times \sigma'(a_1)$
   — 输出层的责任 × 连接权重 × 隐藏层激活函数的灵敏度

3. 计算梯度（权重该如何调整）：
   - $\nabla W_2 = a_1^T \cdot \delta_2$
   - $\nabla W_1 = X^T \cdot \delta_1$

In [7]:
def backward(X, y, a1, a2, W2):
    """
    反向传播：计算每个权重关于 MSE 损失的精确梯度。
    返回 (dW1, db1, dW2, db2)。
    """
    n = len(X)

    # 输出层：dL/dz2（MSE 对 z2 的梯度，含 2/n 因子）
    delta2 = (2.0 / n) * (a2 - y) * sigmoid_deriv(a2)   # shape: (4, 1)

    # 输出层权重的梯度（对所有样本求和）
    dW2 = a1.T @ delta2          # shape: (2, 1)
    db2 = delta2.sum(axis=0, keepdims=True)   # shape: (1, 1)

    # 隐藏层：把输出层的责任往回传（链式法则）
    delta1 = (delta2 @ W2.T) * sigmoid_deriv(a1)   # shape: (4, 2)

    # 隐藏层权重的梯度
    dW1 = X.T @ delta1    # shape: (2, 2)
    db1 = delta1.sum(axis=0, keepdims=True)   # shape: (1, 2)

    return dW1, db1, dW2, db2

# 测试：梯度形状应与权重形状一致
a1, a2 = forward(X, W1, b1, W2, b2)
dW1, db1_grad, dW2, db2_grad = backward(X, y, a1, a2, W2)
print('梯度形状验证：')
print(f'  dW1: {dW1.shape}  （应为 {W1.shape}）')
print(f'  db1: {db1_grad.shape}  （应为 {b1.shape}）')
print(f'  dW2: {dW2.shape}  （应为 {W2.shape}）')
print(f'  db2: {db2_grad.shape}  （应为 {b2.shape}）')


梯度形状验证：
  dW1: (2, 2)  （应为 (2, 2)）
  db1: (1, 2)  （应为 (1, 2)）
  dW2: (2, 1)  （应为 (2, 1)）
  db2: (1, 1)  （应为 (1, 1)）


## Cell 8：训练循环——重复 2000 次

每一轮：
1. 前向传播，得到预测
2. 计算损失
3. 反向传播，得到梯度
4. 用梯度更新权重（学习率 × 梯度）

In [8]:
# 重新初始化（保证可重复）
np.random.seed(0)
W1 = np.random.randn(2, 2) * 0.5
b1 = np.zeros((1, 2))
W2 = np.random.randn(2, 1) * 0.5
b2 = np.zeros((1, 1))

lr = 1.0          # 学习率
epochs = 3000     # 训练轮数
loss_history = []

for epoch in range(epochs):
    # 前向传播
    a1, a2 = forward(X, W1, b1, W2, b2)

    # 计算损失
    loss = mse_loss(a2, y)
    loss_history.append(loss)

    # 反向传播
    dW1, db1_g, dW2, db2_g = backward(X, y, a1, a2, W2)

    # 权重更新（梯度下降：往损失减少的方向走一小步）
    W1 -= lr * dW1
    b1 -= lr * db1_g
    W2 -= lr * dW2
    b2 -= lr * db2_g

    # 每 1000 轮打印一次进度
    if (epoch + 1) % 1000 == 0:
        print(f'第 {epoch+1:4d} 轮 | 损失: {loss:.4f}')

print(f'\n训练完成！最终损失: {loss_history[-1]:.6f}')


第 1000 轮 | 损失: 0.0204
第 2000 轮 | 损失: 0.0028
第 3000 轮 | 损失: 0.0014

训练完成！最终损失: 0.001402


## Cell 9：结果——XOR 全部答对了吗？

In [9]:
_, y_pred = forward(X, W1, b1, W2, b2)

print('训练后的预测结果：')
print('输入        预测值     四舍五入   真实标签')
all_correct = True
for i in range(len(X)):
    pred_raw  = y_pred[i][0]
    pred_bin  = int(round(pred_raw))
    true_label = int(y[i][0])
    correct   = '✓' if pred_bin == true_label else '✗'
    print(f'  {X[i]}  →  {pred_raw:.4f}    {pred_bin}        {true_label}  {correct}')
    if pred_bin != true_label:
        all_correct = False

print()
if all_correct:
    print('结论：两层网络 + 反向传播，XOR 4 个样本全部答对！')
    print('感知机学了 500 轮都失败，这里 2000 轮全对。')
else:
    print('有样本预测错误，可能需要更多训练轮数或调整学习率。')

训练后的预测结果：
输入        预测值     四舍五入   真实标签
  [0. 0.]  →  0.0398    0        0  ✓
  [0. 1.]  →  0.9646    1        1  ✓
  [1. 0.]  →  0.9639    1        1  ✓
  [1. 1.]  →  0.0382    0        0  ✓

结论：两层网络 + 反向传播，XOR 4 个样本全部答对！
感知机学了 500 轮都失败，这里 2000 轮全对。


## Cell 10：可视化——损失曲线

In [10]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, epochs + 1), loss_history, color='steelblue', linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Two-layer network on XOR: loss decreases to near 0', fontsize=13)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=100)
plt.close()
print('损失曲线已保存为 loss_curve.png')
print(f'初始损失: {loss_history[0]:.4f}')
print(f'最终损失: {loss_history[-1]:.6f}')
print(f'下降倍数: {loss_history[0] / loss_history[-1]:.0f}x')

损失曲线已保存为 loss_curve.png
初始损失: 0.2545
最终损失: 0.001402
下降倍数: 182x


## 总结

| | 感知机（节点 02） | 两层网络（本节） |
|--|--|--|
| XOR 能学会吗？ | ❌ 永远失败 | ✅ 2000 轮全对 |
| 层数 | 1 层 | 2 层（含隐藏层） |
| 激活函数 | 阶跃（不可导） | Sigmoid（可导）|
| 学习方法 | 感知机规则 | 反向传播（链式法则） |

**关键突破**：隐藏层 + 链式法则 = 多层网络终于能被训练了。

---
*参考：Rumelhart, Hinton & Williams (1986), Nature 323:533–536. DOI: 10.1038/323533a0*